# 04 · Monthly refresh with Kafka + Spark Structured Streaming (E2)

**Problem:** every month VAHAN publishes new numbers, and the latest month is only partial until it ends. Rebuilding Bronze → Silver → Gold from scratch each time is slow and wasteful.

**Solution:** a streaming pipeline.

```
 new VAHAN file ──► PRODUCER ──► KAFKA topic "vahan.registrations" ──► SPARK STREAMING ──► checks ──► Gold updated
 (bronze folder)    (Step 4)     (a post office for data, in Docker)    (Step 5, runs in the     │
                                                                         background)              └─► bad rows → quarantine
```

- **Producer:** reads a VAHAN file and posts every *state × month* number as one small message (JSON) to Kafka.
- **Kafka:** keeps the messages safely, in order, until someone reads them.
- **Spark Structured Streaming:** keeps watching the topic. For each new *micro-batch* of messages it
  1. runs the same quality rules as Silver (known state, real month, a number, not negative, known fuel group),
  2. sends bad rows to a **quarantine** table instead of Gold,
  3. **upserts** the good rows into `fact_registration_fuel_month`: new month → added, changed month → replaced, same → left alone. Only the affected *year × state* folders are rewritten,
  4. rebuilds the monthly Gold views and the Power BI CSV files,
  5. writes a log line with counts and **latency** (seconds from Kafka to Gold) — our E2 numbers.

**Demos in this notebook**
| Demo | What we send | What should happen |
|---|---|---|
| A · replay | the latest month again (432 messages) | 432 good, 0 changed: Gold unchanged (**idempotent**: running twice is safe) |
| C · bad data | 50 good + 50 deliberately broken messages | all 50 broken ones caught in quarantine, Gold unchanged |
| D · change + undo | Kerala EV Sep 2026 +100, then the original | 1 changed, 1 folder rewritten, views + CSV updated, then back to start |
| B · real refresh | the re-downloaded 2026 files (later, near 15 Nov) | October changed (now complete), November added |

Before you start: **Docker Desktop must be running** ("Engine running"). The first run downloads Kafka (about 400 MB) and the Spark–Kafka connector, so allow 5 minutes.

## Step 1 · Start Kafka in Docker and create the topic

In [1]:
import os, sys, re, csv, time, json, shutil, subprocess, random
from pathlib import Path
from datetime import datetime, date

BASE = Path.cwd().parent                       # ...\Desktop\IAIP
RAW, WH = BASE / "bronze", BASE / "warehouse"
COMPOSE = BASE / "streaming" / "docker-compose.yml"
TOPIC, BOOTSTRAP, CONTAINER = "vahan.registrations", "localhost:9092", "iaip-kafka"
assert (WH / "gold" / "fact_registration_fuel_month").exists(), "Run 03_gold first."
assert shutil.which("docker"), "docker not found: start Docker Desktop, then restart VS Code."

def sh(args, show=True):
    r = subprocess.run(args, capture_output=True, text=True)
    if show:
        print((r.stdout + r.stderr).strip()[-1500:])
    return r

def kafka_cli(tool, *args, show=True):
    return sh(["docker", "exec", CONTAINER, f"/opt/kafka/bin/{tool}", "--bootstrap-server", "localhost:9092", *args], show)

sh(["docker", "compose", "-f", str(COMPOSE), "up", "-d"])          # same as typing it in PowerShell

for i in range(40):                                                 # wait until the broker answers
    if kafka_cli("kafka-topics.sh", "--list", show=False).returncode == 0:
        print(f"Kafka is ready after {i*3} s"); break
    time.sleep(3)
else:
    raise RuntimeError("Kafka did not start. In PowerShell run:  docker logs iaip-kafka")

kafka_cli("kafka-topics.sh", "--create", "--if-not-exists", "--topic", TOPIC, "--partitions", "3", "--replication-factor", "1")
kafka_cli("kafka-topics.sh", "--describe", "--topic", TOPIC)

Container iaip-kafka Starting 
 Container iaip-kafka Started
Kafka is ready after 0 s
Topic: vahan.registrations	TopicId: WT5_QpsHTAuqaRjxl22V3g	PartitionCount: 3	ReplicationFactor: 1	Configs: 
	Topic: vahan.registrations	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: vahan.registrations	Partition: 1	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: vahan.registrations	Partition: 2	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:


CompletedProcess(args=['docker', 'exec', 'iaip-kafka', '/opt/kafka/bin/kafka-topics.sh', '--bootstrap-server', 'localhost:9092', '--describe', '--topic', 'vahan.registrations'], returncode=0, stdout='Topic: vahan.registrations\tTopicId: WT5_QpsHTAuqaRjxl22V3g\tPartitionCount: 3\tReplicationFactor: 1\tConfigs: \n\tTopic: vahan.registrations\tPartition: 0\tLeader: 1\tReplicas: 1\tIsr: 1\tElr: \tLastKnownElr: \n\tTopic: vahan.registrations\tPartition: 1\tLeader: 1\tReplicas: 1\tIsr: 1\tElr: \tLastKnownElr: \n\tTopic: vahan.registrations\tPartition: 2\tLeader: 1\tReplicas: 1\tIsr: 1\tElr: \tLastKnownElr: \n', stderr='')

## Step 2 · Start Spark with the Kafka connector

Spark does not talk to Kafka out of the box. `spark.jars.packages` downloads the official connector (`spark-sql-kafka`) that matches your Spark version. This happens only the first time.

In [2]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
import pyspark
from pyspark.sql import SparkSession, functions as F, types as T, Window

spark = (SparkSession.builder
         .appName("IAIP-04-Streaming")
         .master("local[*]")
         .config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_2.13:{pyspark.__version__}")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.ansi.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "with Kafka connector")

G, SILVER, STREAM = WH / "gold", WH / "silver", WH / "stream"
FACT = G / "fact_registration_fuel_month"
FACT_COLS = ["date_key", "vehicle_type", "fuel_group", "registrations", "is_partial_month", "year", "state_code"]
KEYS = ["date_key", "vehicle_type", "fuel_group", "state_code"]
FUELS = ["EV", "PETROL", "DIESEL", "HYBRID", "GAS", "OTHER"]

VAHAN_MAP = (spark.read.parquet((SILVER / "state_name_map").as_uri())
             .filter("source = 'VAHAN'").select(F.col("raw_name").alias("state_raw"), "state_code")).cache()
spark.read.parquet((G / "dim_state").as_uri()).createOrReplaceTempView("dim_state")

def fact_total():
    return spark.read.parquet(FACT.as_uri()).agg(F.sum("registrations")).first()[0]

START_TOTAL = fact_total()
print(f"Gold fact total before streaming: {START_TOTAL:,}  ({VAHAN_MAP.count()} state spellings known)")

c:\Users\Arka\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark 4.2.0 with Kafka connector
Gold fact total before streaming: 170,025,831  (36 state spellings known)


## Step 3 · The PRODUCER: turn a VAHAN file into Kafka messages

Each message is one number, for example:
```json
{"source_file": "fuel_EV_T_2026.csv", "vehicle_type": "T", "fuel_group": "EV", "state_raw": "Kerala",
 "month_label": "2026-Oct", "value_raw": "1,234", "downloaded_on": "2026-10-30", "sent_at": "...", "batch_label": "demo A"}
```
The **key** (`T|EV|Kerala|2026-Oct`) tells Kafka which partition to use, so updates for the same cell always stay in order.

In [3]:
MSG_SCHEMA = T.StructType([T.StructField(c, T.StringType()) for c in
    ["source_file", "vehicle_type", "fuel_group", "state_raw", "month_label", "value_raw", "downloaded_on", "sent_at", "batch_label"]])
FUEL_FILE = re.compile(r"^fuel_([A-Z]+)_(T|NT)_(\d{4}(?:_\d{4})?)\.csv$")

def read_fuel_file(path):
    """One VAHAN state x month file (layout A) -> list of messages (one per state x month)."""
    m = FUEL_FILE.match(path.name)
    fuel_group, vtype = m.group(1), m.group(2)
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.reader(f))
    months = rows[0][1:]                                   # line 1 = description + month headers
    downloaded_on = date.fromtimestamp(path.stat().st_mtime).isoformat()
    out = []
    for r in rows[1:]:
        if not r or not r[0].strip() or r[0].strip() in ("State", "Total"):
            continue
        for label, v in zip(months, r[1:]):
            if label != "Total":
                out.append({"source_file": path.name, "vehicle_type": vtype, "fuel_group": fuel_group,
                            "state_raw": r[0], "month_label": label, "value_raw": v, "downloaded_on": downloaded_on})
    return out

def files_2026():
    return [p for p in sorted((RAW / "vahan_state_month").glob("fuel_*_2026.csv"))
            if FUEL_FILE.match(p.name).group(1) != "ALL"]          # ALL = sum of the 6 groups, not needed

def publish(records, label):
    """Send messages to Kafka (Spark writes them as a normal batch job)."""
    sent_at = datetime.now().isoformat(timespec="seconds")
    rows = [tuple({**r, "sent_at": sent_at, "batch_label": label}.get(c) for c in MSG_SCHEMA.fieldNames()) for r in records]
    df = spark.createDataFrame(rows, MSG_SCHEMA)
    (df.select(F.concat_ws("|", "vehicle_type", "fuel_group", "state_raw", "month_label").alias("key"),
               F.to_json(F.struct(*MSG_SCHEMA.fieldNames())).alias("value"))
       .write.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP).option("topic", TOPIC).save())
    print(f"published {len(rows):,} messages  [{label}]")

def topic_size():
    r = sh(["docker", "exec", CONTAINER, "/opt/kafka/bin/kafka-get-offsets.sh",
            "--bootstrap-server", "localhost:9092", "--topic", TOPIC], show=False)
    return sum(int(l.rsplit(":", 1)[1]) for l in r.stdout.split() if l.count(":") >= 2)

FILES = files_2026()
LATEST = [l for l in read_fuel_file(FILES[0])][-1]["month_label"]
print(len(FILES), "files for 2026:", ", ".join(p.name for p in FILES))
print("latest month in the files:", LATEST, "| messages already in Kafka:", topic_size())

12 files for 2026: fuel_DIESEL_NT_2026.csv, fuel_DIESEL_T_2026.csv, fuel_EV_NT_2026.csv, fuel_EV_T_2026.csv, fuel_GAS_NT_2026.csv, fuel_GAS_T_2026.csv, fuel_HYBRID_NT_2026.csv, fuel_HYBRID_T_2026.csv, fuel_OTHER_NT_2026.csv, fuel_OTHER_T_2026.csv, fuel_PETROL_NT_2026.csv, fuel_PETROL_T_2026.csv
latest month in the files: 2026-Nov | messages already in Kafka: 1066


## Step 4 · The CONSUMER: Spark Structured Streaming with `foreachBatch`

`process_batch` runs once for every micro-batch of new messages. Read it top to bottom; it is the whole refresh.

**Upsert without a database:** Parquet files cannot be edited row by row. So for each affected *year × state* folder we take the old rows, remove the ones that the new batch replaces, add the new rows, and rewrite **only those folders** (`partitionOverwriteMode = dynamic`). Kerala 2026 can be refreshed without touching Kerala 2021 or any other state. This is why the fact table is partitioned.

In [4]:
def to_number(c):
    c = F.trim(F.regexp_replace(c, ",", ""))
    return F.when(c.isNull() | c.isin("", "NA", "-"), F.lit(None)).otherwise(c.cast("double"))

def gold_sql(name, sql):
    spark.sql(sql).write.mode("overwrite").parquet((G / name).as_uri())
    spark.read.parquet((G / name).as_uri()).createOrReplaceTempView(name)

def rebuild_monthly_gold():
    """Re-create the Gold tables that depend on the monthly fact (same SQL as notebook 03)."""
    spark.read.parquet(FACT.as_uri()).createOrReplaceTempView("fact_registration_fuel_month")
    gold_sql("dim_date", """
        SELECT date_key, month_date, year(month_date) AS year, month(month_date) AS month,
               date_format(month_date, 'MMM') AS month_name, quarter(month_date) AS quarter,
               CASE WHEN month(month_date) >= 4 THEN year(month_date) ELSE year(month_date) - 1 END AS fy_start_year,
               concat('FY', CASE WHEN month(month_date) >= 4 THEN year(month_date) ELSE year(month_date) - 1 END, '-',
                      lpad(CAST((CASE WHEN month(month_date) >= 4 THEN year(month_date) ELSE year(month_date) - 1 END) % 100 + 1 AS STRING), 2, '0')) AS financial_year,
               is_partial_month
        FROM (SELECT date_key, to_date(CAST(date_key AS STRING), 'yyyyMM') AS month_date, bool_or(is_partial_month) AS is_partial_month
              FROM fact_registration_fuel_month GROUP BY date_key) d""")
    spark.sql("""CREATE OR REPLACE TEMP VIEW fm_expanded AS
        WITH typed AS (SELECT state_code, date_key, year, vehicle_type, fuel_group, registrations FROM fact_registration_fuel_month
                       UNION ALL SELECT state_code, date_key, year, 'ALL', fuel_group, registrations FROM fact_registration_fuel_month)
        SELECT * FROM typed UNION ALL SELECT 'IN', date_key, year, vehicle_type, fuel_group, registrations FROM typed""")
    gold_sql("state_month_registrations", """
        SELECT f.state_code, COALESCE(s.state_name, 'All India') AS state_name, s.region,
               d.month_date, d.year, d.month, d.financial_year, f.vehicle_type,
               SUM(f.registrations) AS registrations, d.is_partial_month
        FROM fm_expanded f JOIN dim_date d ON f.date_key = d.date_key LEFT JOIN dim_state s ON f.state_code = s.state_code
        GROUP BY f.state_code, s.state_name, s.region, d.month_date, d.year, d.month, d.financial_year, f.vehicle_type, d.is_partial_month""")
    gold_sql("ev_share_state_month", """
        SELECT f.state_code, COALESCE(s.state_name, 'All India') AS state_name, d.month_date, d.year, d.month, f.vehicle_type,
               SUM(f.registrations) AS total_registrations,
               SUM(CASE WHEN f.fuel_group = 'EV' THEN f.registrations ELSE 0 END) AS ev_registrations,
               SUM(CASE WHEN f.fuel_group = 'HYBRID' THEN f.registrations ELSE 0 END) AS hybrid_registrations,
               ROUND(100 * SUM(CASE WHEN f.fuel_group = 'EV' THEN f.registrations ELSE 0 END) / NULLIF(SUM(f.registrations), 0), 3) AS ev_share_pct,
               d.is_partial_month
        FROM fm_expanded f JOIN dim_date d ON f.date_key = d.date_key LEFT JOIN dim_state s ON f.state_code = s.state_code
        GROUP BY f.state_code, s.state_name, d.month_date, d.year, d.month, f.vehicle_type, d.is_partial_month""")
    gold_sql("fuel_mix_trend", """
        SELECT f.state_code, COALESCE(s.state_name, 'All India') AS state_name, f.year, f.vehicle_type, f.fuel_group,
               SUM(f.registrations) AS registrations,
               ROUND(100 * SUM(f.registrations) / SUM(SUM(f.registrations)) OVER (PARTITION BY f.state_code, f.year, f.vehicle_type), 3) AS share_pct
        FROM fm_expanded f LEFT JOIN dim_state s ON f.state_code = s.state_code
        GROUP BY f.state_code, s.state_name, f.year, f.vehicle_type, f.fuel_group""")
    # refresh the Power BI files of these tables
    tmp, pbi = G / "_tmp_csv", G / "_powerbi"
    for name in ["dim_date", "fact_registration_fuel_month", "state_month_registrations", "ev_share_state_month", "fuel_mix_trend"]:
        src = FACT if name == "fact_registration_fuel_month" else G / name
        spark.read.parquet(src.as_uri()).coalesce(1).write.mode("overwrite").option("header", True).csv((tmp / name).as_uri())
        os.replace(next((tmp / name).glob("part-*.csv")), pbi / f"{name}.csv")
    shutil.rmtree(tmp, ignore_errors=True)

LOG_SCHEMA = ("batch_id long, labels string, received long, good long, bad long, new_rows long, changed_rows long, "
              "unchanged_rows long, folders_rewritten long, months string, gold_updated boolean, "
              "first_msg_in_kafka string, gold_ready_at string, latency_s double")

def process_batch(df, batch_id):
    received = df.count()
    if received == 0:
        return
    df = df.persist()
    first_ts = df.agg(F.min("kafka_ts")).first()[0]

    # 1) the Silver quality rules, applied to every message
    v = (df.join(F.broadcast(VAHAN_MAP), "state_raw", "left")
           .withColumn("month_date", F.to_date(F.concat(F.col("month_label"), F.lit("-01")), "yyyy-MMM-dd"))
           .withColumn("num", to_number(F.col("value_raw")))
           .withColumn("problem",
               F.when(F.col("state_code").isNull(), "unknown state")
                .when(F.col("month_date").isNull(), "bad month label")
                .when(F.col("fuel_group").isNull() | ~F.col("fuel_group").isin(*FUELS), "unknown fuel group")
                .when(F.col("vehicle_type").isNull() | ~F.col("vehicle_type").isin("T", "NT"), "unknown vehicle type")
                .when(F.col("num").isNull() & (F.col("value_raw").isNull() | F.trim("value_raw").isin("", "NA", "-")), "missing value")
                .when(F.col("num").isNull(), "not a number")
                .when(F.col("num") < 0, "negative value")
                .when(F.col("num") != F.floor("num"), "not a whole number"))).persist()

    # 2) bad rows -> quarantine (kept for inspection, never reach Gold)
    bad = v.filter(F.col("problem").isNotNull())
    n_bad = bad.count()
    if n_bad:
        (bad.select(F.lit(batch_id).alias("batch_id"), "problem", *MSG_SCHEMA.fieldNames(), "kafka_ts", "partition", "offset")
            .write.mode("append").parquet((STREAM / "quarantine").as_uri()))

    # 3) good rows: if the same cell arrives twice, keep the newest message
    w = Window.partitionBy("state_code", "month_date", "vehicle_type", "fuel_group").orderBy(F.col("kafka_ts").desc(), F.col("offset").desc())
    new = (v.filter(F.col("problem").isNull())
             .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
             .select(F.date_format("month_date", "yyyyMM").cast("int").alias("date_key"), "vehicle_type", "fuel_group",
                     F.col("num").cast("long").alias("registrations"),
                     (F.col("month_date") >= F.trunc(F.to_date("downloaded_on"), "month")).alias("is_partial_month"),
                     F.year("month_date").alias("year"), "state_code")).persist()
    n_good = new.count()

    # 4) compare with Gold: what is new, what changed, what is the same?
    fact = spark.read.parquet(FACT.as_uri())
    parts = new.select("year", "state_code").distinct()
    old = fact.join(F.broadcast(parts), ["year", "state_code"], "left_semi")
    cmp = new.join(old.select(*KEYS, F.col("registrations").alias("old_reg"), F.col("is_partial_month").alias("old_partial")), KEYS, "left")
    n_new = cmp.filter(F.col("old_reg").isNull()).count()
    n_changed = cmp.filter(F.col("old_reg").isNotNull() & ((F.col("old_reg") != F.col("registrations")) |
                                                           (F.col("old_partial") != F.col("is_partial_month")))).count()

    # 5) upsert: rewrite only the affected year x state folders, then rebuild the monthly Gold views
    changed = n_new + n_changed > 0
    if changed:
        merged = (old.join(new.select(*KEYS), KEYS, "left_anti").select(*FACT_COLS)
                     .unionByName(new.select(*FACT_COLS))
                     .localCheckpoint(eager=True))          # detach from the files we are about to overwrite
        (merged.write.mode("overwrite").option("partitionOverwriteMode", "dynamic")
               .partitionBy("year", "state_code").parquet(FACT.as_uri()))
        rebuild_monthly_gold()

    # 6) log line: the E2 numbers
    labels = ", ".join(sorted(r[0] for r in df.select("batch_label").distinct().collect() if r[0]))
    months = ", ".join(sorted(r[0] for r in v.filter("problem IS NULL").select("month_label").distinct().collect()))
    done = datetime.now()
    row = (batch_id, labels, received, n_good, n_bad, n_new, n_changed, n_good - n_new - n_changed,
           parts.count() if changed else 0, months[:300], changed,
           str(first_ts), done.strftime("%Y-%m-%d %H:%M:%S"), round((done - first_ts).total_seconds(), 1))
    spark.createDataFrame([row], LOG_SCHEMA).write.mode("append").parquet((STREAM / "_stream_log").as_uri())
    for d in (df, v, new):
        d.unpersist()

## Step 5 · Start the stream (it keeps running in the background)

- `startingOffsets = earliest` + a **checkpoint** folder: Spark remembers which messages it has already processed, so after a restart it continues where it stopped and never processes a message twice.
- `trigger(processingTime="5 seconds")`: look for new messages every 5 seconds.
- Set `RESET = True` only if you want Spark to forget its position and re-read every message in the topic. That is safe, because the upsert is idempotent.

In [5]:
RESET = False
CHECKPOINT = STREAM / "_checkpoint"
if RESET:
    shutil.rmtree(CHECKPOINT, ignore_errors=True)

messages = (spark.readStream.format("kafka")
            .option("kafka.bootstrap.servers", BOOTSTRAP)
            .option("subscribe", TOPIC)
            .option("startingOffsets", "earliest")
            .option("failOnDataLoss", "false")
            .load()
            .select(F.from_json(F.col("value").cast("string"), MSG_SCHEMA).alias("m"),
                    F.col("timestamp").alias("kafka_ts"), "partition", "offset")
            .select("m.*", "kafka_ts", "partition", "offset"))

query = (messages.writeStream
         .queryName("vahan_monthly_refresh")
         .foreachBatch(process_batch)
         .option("checkpointLocation", CHECKPOINT.as_uri())
         .trigger(processingTime="5 seconds")
         .start())

query.processAllAvailable()        # first catch up with anything already in the topic
print("stream running:", query.isActive, "|", query.status["message"])

def show_log(n=10):
    p = STREAM / "_stream_log"
    if p.exists():
        (spark.read.parquet(p.as_uri()).orderBy(F.col("batch_id").desc()).limit(n)
              .select("batch_id", "labels", "received", "good", "bad", "new_rows", "changed_rows", "unchanged_rows",
                      "folders_rewritten", "gold_updated", "latency_s").show(truncate=40))

stream running: True | Waiting for data to arrive


## Step 6 · Demo A: replay the latest month (idempotency test)

We send the latest month (October 2026) again from all 12 files (6 fuel groups × T/NT): 36 states × 12 = **432 messages**.
The numbers are identical to what Gold already has, so the correct result is: **432 good, 0 bad, 0 new, 0 changed, Gold total unchanged.** A pipeline where repeating a load changes nothing is called *idempotent*.

In [6]:
recs = [r for p in FILES for r in read_fuel_file(p) if r["month_label"] == LATEST]
publish(recs, "demo A: replay " + LATEST)
query.processAllAvailable()          # wait until Spark has processed everything in the topic
show_log(3)
after = fact_total()
print(f"Gold total before {START_TOTAL:,} | after {after:,} | unchanged: {after == START_TOTAL}")

published 432 messages  [demo A: replay 2026-Oct]
+--------+---------------------------+--------+----+---+--------+------------+--------------+-----------------+------------+---------+
|batch_id|                     labels|received|good|bad|new_rows|changed_rows|unchanged_rows|folders_rewritten|gold_updated|latency_s|
+--------+---------------------------+--------+----+---+--------+------------+--------------+-----------------+------------+---------+
|       6|    demo A: replay 2026-Oct|     288| 288|  0|       0|           0|           288|                0|       false|     21.4|
|       5|    demo A: replay 2026-Oct|     144| 144|  0|       0|           0|           144|                0|       false|     11.5|
|       4|demo C: 50 good + 50 broken|      40|  24| 16|       0|           0|            24|                0|       false|     18.1|
+--------+---------------------------+--------+----+---+--------+------------+--------------+-----------------+------------+---------+

Gold

## Step 7 · Demo C: send bad data on purpose

We take 100 real messages from September 2026. We leave 50 untouched and break the other 50 in five different ways (10 each):

| Broken how | Example |
|---|---|
| unknown state | `Atlantis` |
| bad month label | `2026-Foo` |
| not a number | `12x4` |
| negative value | `-250` |
| unknown fuel group | `COAL` |

Correct result: **50 caught in quarantine, 50 good with 0 changed, Gold total unchanged.** This is the streaming part of E3; the full E3 test comes later.

In [7]:
random.seed(231)
month = LATEST[:5] + "Sep" if LATEST.endswith("Oct") else LATEST
pool = [r for p in FILES for r in read_fuel_file(p) if r["month_label"] == month]
sample = random.sample(pool, 100)
good_part, bad_part = sample[:50], [dict(r) for r in sample[50:]]
BREAKS = [("state_raw", "Atlantis"), ("month_label", "2026-Foo"), ("value_raw", "12x4"),
          ("value_raw", "-250"), ("fuel_group", "COAL")]
for i, r in enumerate(bad_part):
    field, wrong = BREAKS[i // 10]
    r[field] = wrong

LABEL_C = f"demo C: 50 good + 50 broken ({datetime.now():%d-%b %H:%M:%S})"   # unique label per run
publish(good_part + bad_part, LABEL_C)
query.processAllAvailable()
show_log(3)

q = spark.read.parquet((STREAM / "quarantine").as_uri()).filter(F.col("batch_label") == LABEL_C)
q.groupBy("problem").count().orderBy("problem").show(truncate=False)
caught = q.select("partition", "offset").distinct().count()      # one row per broken message
print(f"broken messages caught: {caught} of 50 ({100*caught/50:.0f}%) | Gold total unchanged: {fact_total() == START_TOTAL}")

published 100 messages  [demo C: 50 good + 50 broken]
+--------+---------------------------+--------+----+---+--------+------------+--------------+-----------------+------------+---------+
|batch_id|                     labels|received|good|bad|new_rows|changed_rows|unchanged_rows|folders_rewritten|gold_updated|latency_s|
+--------+---------------------------+--------+----+---+--------+------------+--------------+-----------------+------------+---------+
|       8|demo C: 50 good + 50 broken|      48|  18| 30|       0|           0|            18|                0|       false|     18.9|
|       7|demo C: 50 good + 50 broken|      52|  32| 20|       0|           0|            32|                0|       false|     10.5|
|       6|    demo A: replay 2026-Oct|     288| 288|  0|       0|           0|           288|                0|       false|     21.4|
+--------+---------------------------+--------+----+---+--------+------------+--------------+-----------------+------------+---------+



## Step 7b · Demo D: a real change, then undo it (tests the Gold rewrite)

Demos A and C changed nothing, so the code that **rewrites Gold** has not run yet. Here we:
1. send **Kerala · EV · Non-Transport · Sep 2026** with **+100** registrations,
2. check that the fact table, the `ev_share_state_month` view **and** its Power BI CSV file all show +100, and that only **1 folder** (2026 × Kerala) was rewritten,
3. send the original number back, so Gold returns exactly to where it started.

This gives E2 a real *"Gold updated"* latency.

In [8]:
src = RAW / "vahan_state_month" / "fuel_EV_NT_2026.csv"
SEP = LATEST[:5] + "Sep"
orig = next(r for r in read_fuel_file(src) if r["state_raw"] == "Kerala" and r["month_label"] == SEP)
orig_value = int(float(orig["value_raw"].replace(",", "")))
DATE_KEY = int(SEP[:4] + "09")

def kerala_now():
    cell = (spark.read.parquet(FACT.as_uri())
            .filter(f"state_code = 'KL' AND date_key = {DATE_KEY} AND vehicle_type = 'NT' AND fuel_group = 'EV'")
            .first()["registrations"])
    view = (spark.read.parquet((G / "ev_share_state_month").as_uri())
            .filter(f"state_code = 'KL' AND vehicle_type = 'NT' AND year = {SEP[:4]} AND month = 9").first()["ev_registrations"])
    csv_path = G / "_powerbi" / "ev_share_state_month.csv"
    with open(csv_path, encoding="utf-8") as f:
        row = next(r for r in csv.DictReader(f)
                   if r["state_code"] == "KL" and r["vehicle_type"] == "NT" and r["month_date"].startswith(SEP[:4] + "-09"))
    csv_time = datetime.fromtimestamp(csv_path.stat().st_mtime).strftime("%H:%M:%S")
    return cell, view, int(float(row["ev_registrations"])), csv_time

print(f"original value from the VAHAN file: {orig_value:,}")
print("BEFORE  fact | view | Power BI CSV | CSV saved at:", kerala_now())

publish([{**orig, "value_raw": str(orig_value + 100)}], "demo D: Kerala EV +100")
query.processAllAvailable()
print("CHANGED fact | view | Power BI CSV | CSV saved at:", kerala_now())
print(f"Gold total is now {fact_total():,} (expected {START_TOTAL + 100:,})")

publish([orig], "demo D: undo")
query.processAllAvailable()
print("UNDONE  fact | view | Power BI CSV | CSV saved at:", kerala_now())
show_log(2)
print(f"Gold total back to start: {fact_total() == START_TOTAL}")

original value from the VAHAN file: 15,871
BEFORE  fact | view | Power BI CSV | CSV saved at: (15871, 15871, 15871, '21:13:59')
published 1 messages  [demo D: Kerala EV +100]
CHANGED fact | view | Power BI CSV | CSV saved at: (15971, 15971, 15971, '21:45:00')
Gold total is now 170,025,931 (expected 170,025,931)
published 1 messages  [demo D: undo]
UNDONE  fact | view | Power BI CSV | CSV saved at: (15871, 15871, 15871, '21:45:38')
+--------+----------------------+--------+----+---+--------+------------+--------------+-----------------+------------+---------+
|batch_id|                labels|received|good|bad|new_rows|changed_rows|unchanged_rows|folders_rewritten|gold_updated|latency_s|
+--------+----------------------+--------+----+---+--------+------------+--------------+-----------------+------------+---------+
|      10|          demo D: undo|       1|   1|  0|       0|           1|             0|                1|        true|     25.2|
|       9|demo D: Kerala EV +100|       1|   

## Step 8 · Demo B: the real monthly refresh (run near the end, after re-downloading)

**When:** around 10–14 Nov, after November has started appearing in VAHAN.
1. On the VAHAN site, re-download the **12** files `fuel_{EV,PETROL,DIESEL,HYBRID,GAS,OTHER}_{T,NT}_2026.csv` (same filters as before) and **replace** the old ones in `bronze\vahan_state_month\`.
2. Set `RUN_DEMO_B = True` below and run Steps 1–5 again, then this cell.

Expected: October rows **changed** (now complete), November rows **new**, about 12 folders × 36 states rewritten, and the Gold views + Power BI CSVs refreshed automatically.

In [6]:
RUN_DEMO_B = True
if RUN_DEMO_B:
    FILES = files_2026()
    recs = [r for p in FILES for r in read_fuel_file(p)]
    publish(recs, "demo B: refresh 2026 (" + recs[-1]["downloaded_on"] + ")")
    query.processAllAvailable()
    show_log(3)
    spark.read.parquet((G / "ev_share_state_month").as_uri()) \
         .filter("state_code = 'IN' AND vehicle_type = 'ALL' AND year = 2026") \
         .select("month_date", "total_registrations", "ev_share_pct", "is_partial_month").orderBy("month_date").show()
else:
    print("Demo B is switched off until you re-download the 2026 files.")

published 4,752 messages  [demo B: refresh 2026 (2026-11-13)]
+--------+---------------------------------+--------+----+---+--------+------------+--------------+-----------------+------------+---------+
|batch_id|                           labels|received|good|bad|new_rows|changed_rows|unchanged_rows|folders_rewritten|gold_updated|latency_s|
+--------+---------------------------------+--------+----+---+--------+------------+--------------+-----------------+------------+---------+
|      12|demo B: refresh 2026 (2026-11-13)|    3960|3960|  0|     360|        1030|          2570|               36|        true|     88.4|
|      11|demo B: refresh 2026 (2026-11-13)|     792| 792|  0|      72|         258|           462|               36|        true|     42.8|
|      10|                     demo D: undo|       1|   1|  0|       0|           1|             0|                1|        true|     25.2|
+--------+---------------------------------+--------+----+---+--------+------------+--------

## Step 9 · Stream checks and E2 summary

In [7]:
SQC = []
def sqc(check, expected, actual, passed):
    SQC.append((check, str(expected), str(actual), bool(passed)))
    print(("PASS " if passed else "FAIL ") + f"| {check} | expected {expected} | actual {actual}")

fact = spark.read.parquet(FACT.as_uri())
dup = fact.groupBy(KEYS).count().filter("count > 1").count()
sqc("no duplicate cells in the fact after upserts", 0, dup, dup == 0)
n_months = fact.select("date_key").distinct().count()
sqc("fact is complete (36 states x 6 fuels x 2 types per month)", 36 * 6 * 2 * n_months, fact.count(), fact.count() == 36 * 6 * 2 * n_months)
if not RUN_DEMO_B:
    sqc("Gold total unchanged by demos A, C and D (D was undone)", START_TOTAL, fact_total(), fact_total() == START_TOTAL)
neg = fact.filter("registrations < 0 OR registrations IS NULL").count()
sqc("no negative or empty values reached Gold", 0, neg, neg == 0)
idx = spark.read.parquet((G / "dim_date").as_uri()).count()
sqc("dim_date covers every month in the fact", n_months, idx, idx == n_months)

log = spark.read.parquet((STREAM / "_stream_log").as_uri())
print("\nE2 · micro-batch latency (seconds from first message in Kafka to Gold ready)")
log.groupBy("gold_updated").agg(F.count("*").alias("batches"), F.sum("received").alias("messages"),
                                F.round(F.avg("latency_s"), 1).alias("avg_latency_s"),
                                F.max("latency_s").alias("max_latency_s")).show()
print("messages stored in Kafka topic:", topic_size())
spark.createDataFrame(SQC, "check string, expected string, actual string, passed boolean") \
     .write.mode("overwrite").parquet((STREAM / "_stream_checks").as_uri())
print(f"STREAM CHECKS: {sum(c[3] for c in SQC)} of {len(SQC)} passed")

PASS | no duplicate cells in the fact after upserts | expected 0 | actual 0
PASS | fact is complete (36 states x 6 fuels x 2 types per month) | expected 35856 | actual 35856
PASS | no negative or empty values reached Gold | expected 0 | actual 0
PASS | dim_date covers every month in the fact | expected 83 | actual 83

E2 · micro-batch latency (seconds from first message in Kafka to Gold ready)
+------------+-------+--------+-------------+-------------+
|gold_updated|batches|messages|avg_latency_s|max_latency_s|
+------------+-------+--------+-------------+-------------+
|        true|      4|    4754|         45.1|         88.4|
|       false|      8|    1064|         15.5|         21.4|
+------------+-------+--------+-------------+-------------+

messages stored in Kafka topic: 5818
STREAM CHECKS: 4 of 4 passed


## Step 10 · Stop

Stops the stream and Spark. Kafka keeps running in Docker (it uses little memory). To stop it too: `docker compose -f ..\streaming\docker-compose.yml stop` in PowerShell, or the ■ button in Docker Desktop.

In [8]:
query.stop()
spark.stop()
print("stream and Spark stopped")

stream and Spark stopped
